# 🏀 Calculadora de Rango de Puntos NBA 2025-26

**Objetivo:** Calcular el **MIN OVER** y **MAX UNDER** de un enfrentamiento NBA basado en:
- Estadísticas de temporada (`PPG`, `oPPG`) descargadas de Basketball-Reference.
- Último resultado de cada equipo.
- Lógica de desviación media absoluta (`AVEDEV`) de la hoja de cálculo original.

**Fuente de datos:** [Basketball-Reference NBA 2026](https://www.basketball-reference.com/leagues/NBA_2026_standings.html)

**Algoritmo:**
```
Favor  = PPG  ± AVEDEV(PPG,  Ult.favor,  oPPG_rival)
Contra = oPPG ± AVEDEV(oPPG, Ult.contra, PPG_rival)
Total  = Favor + Contra
```

## 1. Instalación de dependencias
Ejecuta esta celda solo si te falta alguna librería.

In [2]:
# !pip install requests beautifulsoup4 pandas lxml

## 2. Importaciones y configuración

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

BASE_URL = 'https://www.basketball-reference.com/leagues/NBA_2026'
SEASON = '2025-26'

print(f'Temporada configurada: {SEASON}')
print(f'Base URL: {BASE_URL}')

Temporada configurada: 2025-26
Base URL: https://www.basketball-reference.com/leagues/NBA_2026


## 3. Descarga de la tabla de clasificación

Extrae `PPG` (PS/G) y `oPPG` (PA/G) de cada equipo desde la tabla de standings.

> **Nota:** Basketball-Reference suele ocultar las tablas dentro de comentarios HTML. La función maneja ambos casos.

In [2]:
def _extraer_tabla(soup, table_id):
    """Busca una tabla por id, incluso si está dentro de un comentario HTML."""
    tabla = soup.find('table', {'id': table_id})
    if tabla is not None:
        return tabla
    # Buscar dentro de comentarios
    for comentario in soup.find_all(string=lambda t: isinstance(t, str) and table_id in t):
        sub_soup = BeautifulSoup(comentario, 'html.parser')
        tabla = sub_soup.find('table', {'id': table_id})
        if tabla is not None:
            return tabla
    return None


def _parsear_standings(tabla, conferencia):
    """Extrae TEAM, PPG, oPPG de una tabla de standings."""
    equipos = []
    filas = tabla.find('tbody').find_all('tr')
    for fila in filas:
        celdas = fila.find_all(['th', 'td'])
        if len(celdas) < 8:
            continue
        nombre = celdas[0].get_text(strip=True)
        if not nombre or 'Conference' in nombre:
            continue
        try:
            ppg = float(celdas[5].get_text(strip=True))
            oppg = float(celdas[6].get_text(strip=True))
        except (ValueError, IndexError):
            continue
        equipos.append({'TEAM': nombre, 'PPG': ppg, 'oPPG': oppg, 'CONF': conferencia})
    return equipos


def descargar_tabla_equipos():
    url = f'{BASE_URL}_standings.html'
    print(f'Descargando: {url}')
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')

    equipos = []
    for conf, table_id in [('Este', 'confs_standings_E'), ('Oeste', 'confs_standings_W')]:
        tabla = _extraer_tabla(soup, table_id)
        if tabla is None:
            print(f'  ⚠️  No se encontró {table_id}')
            continue
        equipos.extend(_parsear_standings(tabla, conf))

    df = pd.DataFrame(equipos).drop_duplicates(subset=['TEAM']).reset_index(drop=True)
    return df


df_equipos = descargar_tabla_equipos()
print(f'\n✅ {len(df_equipos)} equipos cargados.')
df_equipos.head()

Descargando: https://www.basketball-reference.com/leagues/NBA_2026_standings.html

✅ 30 equipos cargados.


,TEAM,PPG,oPPG,CONF
0,Detroit Pistons*,117.8,109.6,Este
1,Boston Celtics*,114.9,107.2,Este
2,New York Knicks*,116.5,110.1,Este
3,Cleveland Cavaliers*,119.5,115.4,Este
4,Toronto Raptors*,114.6,111.8,Este


## 4. Descarga del último resultado de cada equipo

Se descarga el calendario del mes indicado y se guarda el último marcador de cada equipo.

Puedes ajustar `MES` y `AÑO` en la siguiente celda para apuntar a un mes con partidos jugados (ej. abril 2026 para la temporada pasada completa).

In [5]:
# 🔧 Configura aquí el mes/año a consultar
MES = 4
AÑO = 2026

MESES = {1:'january', 2:'february', 3:'march', 4:'april', 5:'may', 6:'june',
         7:'july', 8:'august', 9:'september', 10:'october', 11:'november', 12:'december'}


def descargar_ultimos_resultados(mes=MES, anio=AÑO):
    url = f'{BASE_URL}_games-{MESES[mes]}.html'
    print(f'Descargando: {url}')
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')

    tabla = _extraer_tabla(soup, 'schedule')
    if tabla is None:
        raise ValueError('No se encontró la tabla de calendario.')

    filas = tabla.find('tbody').find_all('tr')
    ultimos = {}
    for fila in filas:
        celdas = fila.find_all(['th', 'td'])
        if len(celdas) < 6:
            continue
        visitante = celdas[2].get_text(strip=True)
        pts_v = celdas[3].get_text(strip=True)
        local = celdas[4].get_text(strip=True)
        pts_l = celdas[5].get_text(strip=True)
        if not pts_v or not pts_l:
            continue
        try:
            pts_v, pts_l = int(pts_v), int(pts_l)
        except ValueError:
            continue
        ultimos[visitante] = {'favor': pts_v, 'contra': pts_l}
        ultimos[local] = {'favor': pts_l, 'contra': pts_v}
    return ultimos


ultimos_resultados = descargar_ultimos_resultados()
print(f'\n✅ {len(ultimos_resultados)} equipos con resultados recientes.')

Descargando: https://www.basketball-reference.com/leagues/NBA_2026_games-april.html

✅ 30 equipos con resultados recientes.


## 5. Función `AVEDEV` (Desviación Media Absoluta)

Equivalente a la función `AVEDEV` de Excel: promedio de las desviaciones absolutas respecto a la media.

In [3]:
def avedev(*valores):
    """Desviación media absoluta (equivalente a AVEDEV de Excel)."""
    n = len(valores)
    if n == 0:
        return 0.0
    media = sum(valores) / n
    return sum(abs(v - media) for v in valores) / n


# Test rápido
print('AVEDEV(112, 112, 121) =', avedev(112, 112, 121))
print('AVEDEV(114.1, 107, 112.1) =', avedev(114.1, 107, 112.1))

AVEDEV(112, 112, 121) = 4.0
AVEDEV(114.1, 107, 112.1) = 2.7111111111111086


## 6. Cálculo del rango MIN OVER / MAX UNDER

Aplica las fórmulas del algoritmo desde **dos perspectivas** (local y visitante) y consolida:
- `MIN_OVER` = mínimo de los dos mínimos
- `MAX_UNDER` = máximo de los dos máximos

In [7]:
def calcular_rango(equipo_local, equipo_visitante, df_equipos, ultimos):
    """Calcula el rango de puntos totales para un enfrentamiento."""
    s_local = df_equipos[df_equipos['TEAM'] == equipo_local]
    s_visit = df_equipos[df_equipos['TEAM'] == equipo_visitante]

    if s_local.empty:
        raise ValueError(f'Sin estadísticas para {equipo_local}')
    if s_visit.empty:
        raise ValueError(f'Sin estadísticas para {equipo_visitante}')
    if equipo_local not in ultimos:
        raise ValueError(f'Sin último resultado para {equipo_local}')
    if equipo_visitante not in ultimos:
        raise ValueError(f'Sin último resultado para {equipo_visitante}')

    ppg_l = s_local.iloc[0]['PPG']
    oppg_l = s_local.iloc[0]['oPPG']
    ppg_v = s_visit.iloc[0]['PPG']
    oppg_v = s_visit.iloc[0]['oPPG']

    ult_l = ultimos[equipo_local]
    ult_v = ultimos[equipo_visitante]

    # --- Perspectiva LOCAL ---
    desv_fav_l = avedev(ppg_l, ult_l['favor'], oppg_v)
    desv_con_l = avedev(oppg_l, ult_l['contra'], ppg_v)
    fav_min_l = ppg_l - desv_fav_l
    fav_max_l = ppg_l + desv_fav_l
    con_min_l = oppg_l - desv_con_l
    con_max_l = oppg_l + desv_con_l
    tmin_l = fav_min_l + con_min_l
    tmax_l = fav_max_l + con_max_l

    # --- Perspectiva VISITANTE ---
    desv_fav_v = avedev(ppg_v, ult_v['favor'], oppg_l)
    desv_con_v = avedev(oppg_v, ult_v['contra'], ppg_l)
    fav_min_v = ppg_v - desv_fav_v
    fav_max_v = ppg_v + desv_fav_v
    con_min_v = oppg_v - desv_con_v
    con_max_v = oppg_v + desv_con_v
    tmin_v = fav_min_v + con_min_v
    tmax_v = fav_max_v + con_max_v

    return {
        'local': equipo_local,
        'visitante': equipo_visitante,
        'ppg_local': ppg_l, 'oppg_local': oppg_l,
        'ppg_visitante': ppg_v, 'oppg_visitante': oppg_v,
        'ult_local': ult_l, 'ult_visitante': ult_v,
        'total_min_local': round(tmin_l, 2),
        'total_max_local': round(tmax_l, 2),
        'total_min_visitante': round(tmin_v, 2),
        'total_max_visitante': round(tmax_v, 2),
        'MIN_OVER': round(min(tmin_l, tmin_v), 2),
        'MAX_UNDER': round(max(tmax_l, tmax_v), 2),
    }

## 7. Selección del enfrentamiento

Ejecuta la siguiente celda y escribe los nombres de los equipos. Para ver la lista exacta de nombres disponibles, ejecuta la celda de ayuda.

In [8]:
# Ver lista de equipos disponibles
print('Equipos disponibles:')
for i, eq in enumerate(sorted(df_equipos['TEAM'].tolist()), 1):
    print(f'  {i:2d}. {eq}')

Equipos disponibles:
   1. Atlanta Hawks*
   2. Boston Celtics*
   3. Brooklyn Nets
   4. Charlotte Hornets
   5. Chicago Bulls
   6. Cleveland Cavaliers*
   7. Dallas Mavericks
   8. Denver Nuggets*
   9. Detroit Pistons*
  10. Golden State Warriors
  11. Houston Rockets*
  12. Indiana Pacers
  13. Los Angeles Clippers
  14. Los Angeles Lakers*
  15. Memphis Grizzlies
  16. Miami Heat
  17. Milwaukee Bucks
  18. Minnesota Timberwolves*
  19. New Orleans Pelicans
  20. New York Knicks*
  21. Oklahoma City Thunder*
  22. Orlando Magic*
  23. Philadelphia 76ers*
  24. Phoenix Suns*
  25. Portland Trail Blazers*
  26. Sacramento Kings
  27. San Antonio Spurs*
  28. Toronto Raptors*
  29. Utah Jazz
  30. Washington Wizards


In [9]:
# 🔧 Escribe aquí el enfrentamiento a analizar
EQUIPO_LOCAL = 'Detroit Pistons'
EQUIPO_VISITANTE = 'Brooklyn Nets'

resultado = calcular_rango(EQUIPO_LOCAL, EQUIPO_VISITANTE, df_equipos, ultimos_resultados)

ValueError: Sin estadísticas para Detroit Pistons

## 8. Resultado del análisis

In [10]:
def imprimir_resultado(r):
    print('=' * 62)
    print(f"  🏀 {r['local']} (LOCAL) vs {r['visitante']} (VISITANTE)")
    print('=' * 62)
    print(f"  {r['local']:25s}  PPG: {r['ppg_local']:.1f}  |  oPPG: {r['oppg_local']:.1f}")
    print(f"  {r['visitante']:25s}  PPG: {r['ppg_visitante']:.1f}  |  oPPG: {r['oppg_visitante']:.1f}")
    print('-' * 62)
    print(f"  Último {r['local']:22s}: {r['ult_local']['favor']} - {r['ult_local']['contra']}")
    print(f"  Último {r['visitante']:22s}: {r['ult_visitante']['favor']} - {r['ult_visitante']['contra']}")
    print('-' * 62)
    print(f"  Perspectiva {r['local']:20s}:  Min {r['total_min_local']:.2f}  |  Max {r['total_max_local']:.2f}")
    print(f"  Perspectiva {r['visitante']:20s}:  Min {r['total_min_visitante']:.2f}  |  Max {r['total_max_visitante']:.2f}")
    print('=' * 62)
    print(f"  🟢 MIN OVER  (línea mínima total): {r['MIN_OVER']:.2f}")
    print(f"  🔴 MAX UNDER (línea máxima total): {r['MAX_UNDER']:.2f}")
    print('=' * 62)
    print(f"  → OVER con valor si la línea < {r['MIN_OVER']:.2f}")
    print(f"  → UNDER con valor si la línea > {r['MAX_UNDER']:.2f}")

imprimir_resultado(resultado)

NameError: name 'resultado' is not defined

## 9. Análisis múltiple (opcional)

Analiza varios enfrentamientos en una sola pasada.

In [11]:
# 🔧 Añade aquí todos los enfrentamientos que quieras analizar
partidos = [
    ('Boston Celtics', 'New York Knicks'),
    ('Oklahoma City Thunder', 'Denver Nuggets'),
    ('Los Angeles Lakers', 'Golden State Warriors'),
]

resumen = []
for local, visitante in partidos:
    try:
        r = calcular_rango(local, visitante, df_equipos, ultimos_resultados)
        resumen.append({
            'LOCAL': local,
            'VISITANTE': visitante,
            'MIN_OVER': r['MIN_OVER'],
            'MAX_UNDER': r['MAX_UNDER'],
            'RANGO': r['MAX_UNDER'] - r['MIN_OVER'],
        })
    except ValueError as e:
        print(f'⚠️  {local} vs {visitante}: {e}')

df_resumen = pd.DataFrame(resumen)
df_resumen

⚠️  Boston Celtics vs New York Knicks: Sin estadísticas para Boston Celtics
⚠️  Oklahoma City Thunder vs Denver Nuggets: Sin estadísticas para Oklahoma City Thunder
⚠️  Los Angeles Lakers vs Golden State Warriors: Sin estadísticas para Los Angeles Lakers


""


## 10. Exportar resultados

Guarda el DataFrame de equipos y el resumen en archivos CSV.

In [12]:
df_equipos.to_csv('nba_2026_equipos.csv', index=False)
if not df_resumen.empty:
    df_resumen.to_csv('nba_2026_rangos.csv', index=False)
    print('✅ Archivos exportados: nba_2026_equipos.csv, nba_2026_rangos.csv')
else:
    print('✅ Archivo exportado: nba_2026_equipos.csv')

✅ Archivo exportado: nba_2026_equipos.csv


---
## Notas finales

### Sobre el algoritmo
- `AVEDEV` actúa como **margen de dispersión** entre el promedio de temporada, el último resultado y el promedio del rival.
- Se calcula desde dos perspectivas (local y visitante) y se consolida con **mínimo más bajo** y **máximo más alto**.
- El rango (`MAX_UNDER - MIN_OVER`) indica la **incertidumbre**: rangos amplios = mayor volatilidad.

### Interpretación para apuestas
- Si la línea de la casa de apuestas está **por debajo de `MIN_OVER`** → **OVER con valor**.
- Si la línea está **por encima de `MAX_UNDER`** → **UNDER con valor**.
- Si la línea cae **dentro del rango** → zona neutral, sin ventaja clara.

### Limitaciones
- No considera lesiones, descanso, back-to-backs ni contexto táctico.
- `VLOOKUP` original asume un único partido a la vez; aquí se ha hecho explícito por equipo.
- Basketball-Reference puede cambiar el HTML: si falla, revisa los `table_id`.